*In this tutorial we will explore how many of Oxford Street's souvenir and candy shops are connected.*

In a [recent article](https://www.ft.com/candy) in the Financial Times, journalists mapped a number of candy shops on Oxford Street and noted that the company officers appear to form a loose network "with some sharing residential or business addresses, or taking ownership of a business for months at a time before ceding to another shareholder". Using the addresses and companies mentioned in the article, I compiled the following relevant records from Companies House:

- CITY SOUVENIRS LTD (539 Oxford Street): 08658716
- London Hot Accessories Limited (537 Oxford Street): 10116914
- AMERICAN SWEET DREAMS LIMITED (524 Oxford Street): 13938312
- Western Crown Limited (470-482 Oxford Street): 13455377
- CANDYLICIOUS WHOLESALE LIMITED (470-472 Oxford Street): 14091125
- FREAKNAUGHTY LTD (407 Oxford Street): 11730327
- NASTY BANG LTD (324 Oxford Street): 14223273
- Quality Products and Merchandise Ltd (321-323 Oxford Street): 14518117
- Kingdom of Sweets (270 Oxford Street): 11004735
- LND Accesorize Limited (271 Oxford Street): 11601607
- Candystreet (146-148 Oxford Street): 12415826
- E & A Accessories Limited (35 Oxford Street): 14261732
- Breeze Vape Limited (33 Oxford Street): 14050986
- Drip Vape (33 Oxford Street): 14055609
- Gift 4 You (4 Oxford Street): 11439227

The following companies were also mentioned however I couldn't find a relevant record in Companies House:

- London Dream (476 Oxford Street)
- Welcome London (399-403 Oxford Street)
- American Candy Shop (385-389 Oxford Street)
- Candy World (363-367 Oxford Street)
- Tobacco & Cigarettes Sold Here (273 Oxford Street)
- Unique Gifts (159 Oxford Street)
- Toys and Gifts (142-144 Oxford Street)
- American Candy World (119-121 Oxford Street)
- American Sweets & Souvenirs (37-39 Oxford Street)

In this tutorial we will build a large network of companies that connect to a single company on Oxford Street. We can use the following dictionary of companies for reference: 

In [ ]:
candy_shops = [{"company_name": "City Souvenirs", "address": "539 Oxford Street", "company_id": "08658716"},
            {"company_name": "London Hot Accessories", "address": "537 Oxford Street", "company_id": "10116914"},
            {"company_name": "American Sweet Dreams", "address": "524 Oxford Street", "company_id": "13938312"},
            {"company_name": "London Dream", "address": "476 Oxford Street", "company_id": ""},
            {"company_name": "Western Crown", "address": "470-482 Oxford Street", "company_id": "13455377"},
            {"company_name": "Candylicious Wholesale", "address": "470-482 Oxford Street", "company_id": "14091125"},
            {"company_name": "Freakynaughty", "address": "407 Oxford Street", "company_id": "11730327"},
            {"company_name": "Welcome London", "address": "399-403 Oxford Street", "company_id": ""},
            {"company_name": "American Candy Shop", "address": "385-389 Oxford Street", "company_id": ""},
            {"company_name": "Candy World", "address": "363-367 Oxford Street", "company_id": ""},
            {"company_name": "Nasty Bang", "address": "324 Oxford Street", "company_id": "14223273"},
            {"company_name": "Quality Products and Merchandise", "address": "324 Oxford Street", "company_id": "14223273"},
            {"company_name": "Tobacco & Cigarettes Sold Here", "address": "273 Oxford Street", "company_id": ""},
            {"company_name": "LND Accesorize", "address": "271 Oxford Street", "company_id": "11601607"},
            {"company_name": "Unique Gifts", "address": "159 Oxford Street", "company_id": ""},
            {"company_name": "Toys and Gifts", "address": "142-144 Oxford Street", "company_id": ""},
            {"company_name": "Candystreet", "address": "146-148 Oxford Street", "company_id": "12415826"},
            {"company_name": "American Candy World", "address": "119-121 Oxford Street", "company_id": ""},
            {"company_name": "American Sweets & Souvenirs", "address": "37-39 Oxford Street", "company_id": ""},
            {"company_name": "E & A Accessories Limited", "address": "35 Oxford Street", "company_id": "14261732"},
            {"company_name": "Breeze Vape Limited", "address": "33 Oxford Street", "company_id": "14050986"},
            {"company_name": "Drip Vape", "address": "33 Oxford Street", "company_id": "14055609"},
            {"company_name": "Gift 4 You", "address": "4 Oxford Street", "company_id": "11439227"}]

In [ ]:
import sugartrail
import pandas as pd
from ipywidgets import HTML, Widget, Layout, Output, VBox, HBox, Textarea
from pathlib import Path
from IPython.display import Image, display

def _find_project_root():
    p = Path.cwd()
    while p != p.parent:
        if (p / 'pyproject.toml').exists():
            return p
        p = p.parent
    return Path.cwd()

_PROJECT_ROOT = _find_project_root()

Add Companies House API key:

In [ ]:
# API key is loaded automatically from .env (COMPANIES_HOUSE_API_KEY).
# To set it manually instead: sugartrail.api.basic_auth.username = "your_key"
if not sugartrail.api.basic_auth.username:
    print("No API key found — add COMPANIES_HOUSE_API_KEY to your .env file")

Lets investigate "Western Crown" which has an id of "13455377"

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / 'western_crown.png')))

In [ ]:
origin_company_id="13455377"
western_crown_network = sugartrail.base.Network(company_id=origin_company_id)

Lets impose some limits on the results. As we're doing a deeper search we want to avoid accumulating lots of irrelevant data connected to incorporation agents and virtual offices. By setting the following limits, the network will not include results that exceed these limits:

In [ ]:
western_crown_network.hop.companies_at_address_maxsize = 50
western_crown_network.hop.officers_at_address_maxsize = 50
western_crown_network.hop.officer_appointments_maxsize = 50

Lets go big and perform 6 hops. It's likely to take some time to gather all the data (1+ hour). If you don't want to wait, you can also use uncomment the block below to load a pre-made network instance, in which case, jump to the 'generate map' cell :

In [ ]:
# western_crown_network = sugartrail.base.Network(file=f'{sugartrail.const.networks_path}candy_connections_a.json')

In [ ]:
western_crown_network.perform_hop(6)

In [ ]:
western_crown_network.run_map_preprocessing()

In [ ]:
# generate map (may take half a minute!)
map_data,path_table = sugartrail.mapvis.build_map(western_crown_network) 
hbox = HBox([path_table])
vbox = VBox([map_data, hbox])
vbox

### Oxford Street Connections

From the data we've gathered, there are many companies and addresses that connect with the original address the network was built from (Western Crown Limited). To print the connections we can pass the company ID to `find_path`:

In [ ]:
pd.DataFrame(western_crown_network.find_path('10895963'))[['node_index', 'title', 'id', 'depth', 'node_type', 'link_type', 'link']]

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '537.png')))

*Active Companies*
- [BEST OF LONDON LIMITED](https://find-and-update.company-information.service.gov.uk/company/10895963) (10895963)

*Dissolved Companies*
- [LONDON HOT ACCESSORIES LIMITED](https://find-and-update.company-information.service.gov.uk/company/10116914) (10116914)
- [TOURISTS WORLD LTD.](https://find-and-update.company-information.service.gov.uk/company/10643744) (10643744)
- [GIFTS FOR TOURIST LIMITED](https://find-and-update.company-information.service.gov.uk/company/10910649) (10910649)

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '524.png')))

*Active Companies*
- [CANDY TOWN LTD](https://find-and-update.company-information.service.gov.uk/company/11464159) (11464159)
- [ESPANZA LIMITED](https://find-and-update.company-information.service.gov.uk/company/11474248) (11474248)

*Dissolved Companies*
- [MARGIN FREE SUPER MARKET LIMITED](https://find-and-update.company-information.service.gov.uk/company/10540083) (10540083)
- [COOL MIX LIMITED](https://find-and-update.company-information.service.gov.uk/company/11031538) (11031538)
- [ROCK GIFTS LTD](https://find-and-update.company-information.service.gov.uk/company/11588633) (11588633)

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '470.png')))

*Active Companies*
- [WESTERN CROWN LIMITED](https://find-and-update.company-information.service.gov.uk/company/13455377) (13455377)

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '447.png')))

*Dissolved Companies*
- [PLANET SOUVENIRS (UK) LIMITED](https://find-and-update.company-information.service.gov.uk/company/07570906) (07570906)

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '407.png')))

*Active Companies*
- [WESTERN CROWN LIMITED](https://find-and-update.company-information.service.gov.uk/company/13455377) (13455377)

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '269.png')))

*Active Companies*
- [MOODY MOON LIMITED](https://find-and-update.company-information.service.gov.uk/company/13287820) (13287820)

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '263.png')))

*Active Companies*
- [CEEKEY LONDON LTD](https://find-and-update.company-information.service.gov.uk/company/11647876) (11647876)

*Liquidated Companies*
- [RUSTIC RAY LTD](https://find-and-update.company-information.service.gov.uk/company/11758349) (11758349)

*Dissolved Companies*
- [LILLY MAX LIMITED](https://find-and-update.company-information.service.gov.uk/company/11474310) (11474310)
- [GIFTNET LTD](https://find-and-update.company-information.service.gov.uk/company/11593230) (11593230)
- [BUMPZ LTD](https://find-and-update.company-information.service.gov.uk/company/10941293) (10941293)
- [VENGAT GIFT LTD](https://find-and-update.company-information.service.gov.uk/company/11647421) (11647421)

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '240.png')))

*Active Companies*
- [GIFT 4 YOU LIMITED](https://find-and-update.company-information.service.gov.uk/company/11439227) (11439227)
- [WEST END MANAGEMENT LIMITED](https://find-and-update.company-information.service.gov.uk/company/11467385) (11467385)

*Dissolved Companies*
- [TOURISTS WORLD LTD.](https://find-and-update.company-information.service.gov.uk/company/10643744) (10643744)
- [GIFTS FOR TOURIST LIMITED](https://find-and-update.company-information.service.gov.uk/company/10910649) (10910649)
- [LILLY MAX LIMITED](https://find-and-update.company-information.service.gov.uk/company/11474310) (11474310)

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '158.png')))

*Dissolved Companies*

- [UNIQUE GIFTS (LONDON) LTD](https://find-and-update.company-information.service.gov.uk/company/07060273) (07060273)

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '146.png')))

*Active Companies*
- [CANDYSTREET LTD](https://find-and-update.company-information.service.gov.uk/company/12415826) (12415826)
- [LUMS SWEETS LTD](https://find-and-update.company-information.service.gov.uk/company/11864536) (11864536)
- [SPARK LABEL LTD](https://find-and-update.company-information.service.gov.uk/company/13865359) (13865359)
- [SEEN CAPTURE LTD](https://find-and-update.company-information.service.gov.uk/company/11468719) (11468719)
- [PINEBIRD LTD](https://find-and-update.company-information.service.gov.uk/company/11869360) (11869360)

*Dissolved Companies*
- [WESTERN CANDIES LTD](https://find-and-update.company-information.service.gov.uk/company/12005109) (12005109)
- [GIFTS OF GLORY LTD](https://find-and-update.company-information.service.gov.uk/company/12268339) (12268339)
- [CANDY CANE LTD](https://find-and-update.company-information.service.gov.uk/company/12005370) (12005370)
- [XEE ASSET MANAGEMENT LTD](https://find-and-update.company-information.service.gov.uk/company/10734212) (10734212)
- [GRAND STORE LTD](https://find-and-update.company-information.service.gov.uk/company/11843228) (11843228)

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '142.png')))

*Active Companies*
- [GRAND EMPIRE LIMITED](https://find-and-update.company-information.service.gov.uk/company/13376158) (13376158)
- [ASUS BLUE LIMITED](https://find-and-update.company-information.service.gov.uk/company/13795800) (3795800)

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '41.png')))

*Active Companies*
- [FANCY SOUVENIRS LIMITED](https://find-and-update.company-information.service.gov.uk/company/05548476) (05548476)

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '37.png')))

*Active Companies*

- [CHOCO-FIVES LTD](https://find-and-update.company-information.service.gov.uk/company/13967497) (13967497)
- [FANCYGREEN LTD](https://find-and-update.company-information.service.gov.uk/company/14210992) (14210992)
- [FORTLEE LTD](https://find-and-update.company-information.service.gov.uk/company/10782536) (10782536)
- [CEEKEY LONDON LTD](https://find-and-update.company-information.service.gov.uk/company/11647876) (11647876)
- [CHERRYTREE FOUNDATION](https://find-and-update.company-information.service.gov.uk/company/08632458) (08632458)
- [CHOCO-LOT LTD](https://find-and-update.company-information.service.gov.uk/company/13964334) (13964334)
- [FABIAN BELL LTD](https://find-and-update.company-information.service.gov.uk/company/11855234) (11855234)
- [SEEN CAPTURE LTD](https://find-and-update.company-information.service.gov.uk/company/11468719) (11468719)
- [ASUSGIFTS LIMITED](https://find-and-update.company-information.service.gov.uk/company/13857378) (13857378)
- [BRITCO GIFTS LIMITED](https://find-and-update.company-information.service.gov.uk/company/14472151) (14472151)
- [AH MONEY EXCHANGE LTD](https://find-and-update.company-information.service.gov.uk/company/10231441) (10231441)

*Dissolved Companies*
- [GIFT PUNCH LIMITED](https://find-and-update.company-information.service.gov.uk/company/12387042) (12387042)
- [GIFTNET LTD](https://find-and-update.company-information.service.gov.uk/company/11593230) (11593230)
- [JUICE COLLECTIVE LTD](https://find-and-update.company-information.service.gov.uk/company/08282993) (08282993)
- [WINHAND LTD](https://find-and-update.company-information.service.gov.uk/company/11016980) (11016980)

In [ ]:
display(Image(str(_PROJECT_ROOT / 'assets' / 'images' / '4.png')))

*Active Companies*
- [GIFT 4 YOU LIMITED](https://find-and-update.company-information.service.gov.uk/company/11439227) (11439227)

### Beyond Oxford Street

From analysing the 'generate map' cell, we will notice many other companies connected to the original company located beyond Oxford Street. Further research could involve:
- building networks from some of the other companies registered to Oxford Street addresses
- building networks deeper than 6 hops to explore a wider range of connections
- analysing connections in greater detail
- analysing documents from Companies House linked to companies in the network
- identify other connected companies of interest beyond Oxford Street
- develop statistics that communicate the scale of these networks and connectivity within the UK
- analyse connections outside the UK 
- run a news search on entities within the network to see if companies are connected to any newsworthy entities
- analyse hotspots for registering new companies over time to see if there are emerging popular locations, in other words where is the new Oxford Street?
- analyse other types of companies connected to souvenir and candy shops (money exchanges, security firms etc.)